# QCNN（Colab 版）

本笔记本用于在 Google Colab 上直接运行本项目，满足以下要求：
- 与本地 `QCNN_project` 虚拟环境一致的依赖（使用项目自带 `requirements.txt` 安装）；
- 在此基础上额外安装并优先使用 `qiskit-aer-gpu` 以利用 Colab 的 GPU；
- 在开头自动挂载 Google Drive。

建议：先将整个项目文件夹放到 Drive 下（默认路径 `MyDrive/QCNN_project`），然后在 Colab 中打开本笔记本并依次运行各单元。


## 1. 挂载 Google Drive
运行后在弹出的窗口中进行授权。

In [ ]:
from google.colab import drive  # type: ignore
drive.mount('/content/drive', force_remount=True)


## 2. 指定/定位项目目录
默认尝试在 `MyDrive/QCNN_project` 下寻找；如未找到请手动修改 `PROJECT_DIR`。

In [ ]:
import os
from pathlib import Path

PROJECT_NAME = 'QCNN_project'
DRIVE_ROOT = Path('/content/drive/MyDrive')
CANDIDATES = [
    DRIVE_ROOT / PROJECT_NAME,
    DRIVE_ROOT / 'Colab Notebooks' / PROJECT_NAME,
]
PROJECT_DIR = next((p for p in CANDIDATES if p.exists()), None)
if PROJECT_DIR is None:
    # 如未找到，请手动设置为你在 Drive 中的实际路径
    PROJECT_DIR = DRIVE_ROOT / PROJECT_NAME
    print('未在默认位置找到项目，请确保路径存在或修改 PROJECT_DIR。')

print(f'PROJECT_DIR = {PROJECT_DIR}')
os.chdir(PROJECT_DIR)
print('工作目录：', os.getcwd())


## 3. 环境就绪：安装依赖 + 启用 qiskit-aer-gpu
- 默认使用项目根目录下的 `requirements.txt`，尽量与本地 `QCNN_project` 环境一致。
- 会自动卸载 CPU 版 `qiskit-aer` 并尝试安装 `qiskit-aer-gpu`（如不可用则回退到 CPU 版）。
- 如果系统已具备 GPU 版 PyTorch，本步骤会优先保留现有 PyTorch，避免覆盖 GPU 加速。

In [ ]:
import sys, subprocess, tempfile
from pathlib import Path

def run(cmd, check=True):
    print('>>>', ' '.join(cmd))
    return subprocess.run(cmd, check=check)

PIP = [sys.executable, '-m', 'pip']
run(PIP + ['install', '-U', 'pip', 'setuptools', 'wheel'])

req = Path('requirements.txt')
if not req.exists():
    raise FileNotFoundError('未找到 requirements.txt，请确认当前工作目录为项目根目录。')

# 读取并过滤依赖：先排除 qiskit-aer（将安装 GPU 版），必要时保留/跳过 PyTorch
raw_lines = req.read_text(encoding='utf-8').splitlines()
lines = []
for line in raw_lines:
    s = line.strip()
    if not s or s.startswith('#'):
        continue
    if 'qiskit-aer' in s.replace(' ', ''):
        # 改为后续安装 GPU 版
        continue
    lines.append(s)

# 若当前环境已具备 GPU 版 PyTorch，则避免覆盖
skip_torch = False
try:
    import torch  # type: ignore
    if torch.cuda.is_available():
        skip_torch = True
except Exception:
    pass

if skip_torch:
    lines = [l for l in lines if not l.strip().startswith('torch') and not l.strip().startswith('torchvision')]

# 将过滤后的依赖写入临时文件并安装
with tempfile.NamedTemporaryFile('w', delete=False) as tmp:
    tmp.write('
'.join(lines))
    tmp_path = tmp.name

run(PIP + ['install', '-r', tmp_path])

# 处理 Qiskit Aer：先卸载 CPU 版，再尝试 GPU 版，不行再回退 CPU 版
run(PIP + ['uninstall', '-y', 'qiskit-aer'], check=False)
gpu_installed = False
for spec in ['qiskit-aer-gpu==0.17.1', 'qiskit-aer-gpu']:
    try:
        run(PIP + ['install', spec])
        gpu_installed = True
        break
    except subprocess.CalledProcessError:
        continue

if not gpu_installed:
    print('!!! 未能安装 GPU 版 Aer，回退安装 CPU 版 qiskit-aer==0.17.1')
    run(PIP + ['install', 'qiskit-aer==0.17.1'])

# 基本检查
import importlib
qiskit_aer = importlib.import_module('qiskit_aer')
from qiskit_aer import AerSimulator
print('qiskit-aer 版本：', getattr(qiskit_aer, '__version__', 'unknown'))

try:
    sim = AerSimulator(method='statevector', device='GPU')
    print('AerSimulator GPU 构造成功 ✅')
except Exception as e:
    print('警告：无法以 GPU 模式初始化 AerSimulator，可能将以 CPU 运行。', e)

# PyTorch 情况
try:
    import torch
    print('PyTorch 版本：', torch.__version__, 'CUDA 可用：', torch.cuda.is_available())
except Exception as e:
    print('PyTorch 未安装或不可用：', e)


## 4. 选择配置并运行训练（可选）
确保你的配置文件 `environment.backend` 为 `GPU` 才能使用 GPU 版 Aer。默认示例会将其自动改为 `GPU`。

In [ ]:
from pathlib import Path
import yaml, subprocess, sys

CONFIG_PATH = Path('configs/mnist_amplitude_quick.yaml')  # 可根据需要修改
FORCE_GPU_BACKEND = True  # 将配置中的 environment.backend 强制改为 'GPU'

if FORCE_GPU_BACKEND and CONFIG_PATH.exists():
    cfg = yaml.safe_load(open(CONFIG_PATH, 'r', encoding='utf-8'))
    env = cfg.setdefault('environment', {})
    env['backend'] = 'GPU'
    with open(CONFIG_PATH, 'w', encoding='utf-8') as f:
        yaml.safe_dump(cfg, f, allow_unicode=True)
    print('已将配置中的 environment.backend 设置为 GPU:', CONFIG_PATH)

print('开始训练...')
subprocess.run([sys.executable, 'train.py', '--config', str(CONFIG_PATH)], check=True)


## 5. 推理评估（加载 checkpoint 并统计准确率）\n使用训练后的 checkpoint 对测试集进行评估，输出准确率。根据需要修改 `CONFIG_PATH` 和 `CKPT_PATH`。\n

In [ ]:
from pathlib import Path
import yaml, torch
from qiskit_aer.primitives import Estimator as AerEstimator
from qiskit_aer.noise import NoiseModel, depolarizing_error
from models.qcnn import QCNNAmplitude, QCNNGeneral
from encoders.angle import build_angle_encoder_circuit
from encoders.hybrid import build_hybrid_encoder_circuit
from train import get_dataloader

CONFIG_PATH = Path('configs/mnist_amplitude_colab.yaml')  # 修改为你的配置
CKPT_PATH = Path('checkpoints/mnist_amplitude/best.pt')  # 修改为你的 checkpoint
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')

config = yaml.safe_load(open(CONFIG_PATH, 'r', encoding='utf-8'))
env = config['environment']
backend_options = {}
if env.get('backend') == 'GPU' and torch.cuda.is_available():
    backend_options['device'] = 'GPU'
if env.get('add_noise'):
    nm = NoiseModel()
    p1 = env.get('noise', {}).get('depolarizing_p1', 0.0)
    p2 = env.get('noise', {}).get('depolarizing_p2', 0.0)
    if p1 > 0: nm.add_all_qubit_quantum_error(depolarizing_error(p1, 1), ['ry','rz','h'])
    if p2 > 0: nm.add_all_qubit_quantum_error(depolarizing_error(p2, 2), ['cx'])
    backend_options['noise_model'] = nm
estimator = AerEstimator(backend_options=backend_options)

data = config['data']; encoding = data['encoding']
if encoding == 'amplitude':
    model = QCNNAmplitude(num_qubits=data['num_qubits'], num_classes=data['num_classes'], estimator=estimator)
else:
    encoder_fn = {'angle': build_angle_encoder_circuit, 'hybrid': build_hybrid_encoder_circuit}[encoding]
    num_input_features = data.get('num_features', data['num_qubits'])
    model = QCNNGeneral(num_qubits=data['num_qubits'], encoder_fn=encoder_fn, num_input_features=num_input_features, num_classes=data['num_classes'], estimator=estimator)
model.to(device)
ckpt = torch.load(CKPT_PATH, map_location=device)
model.load_state_dict(ckpt['model_state'])
model.eval()

test_loader = get_dataloader(config, train=False)
correct, total = 0, 0
with torch.no_grad():
    for x, y in test_loader:
        x, y = x.to(device), y.to(device)
        pred = model(x).argmax(dim=1)
        correct += (pred == y).sum().item()
        total += y.size(0)
acc = 100.0 * correct / total
print(f'Test accuracy: {acc:.2f}% ({correct}/{total})')
